# Lab 07-04 · Laplace-Domain Circuit Analysis

Companion to chapter [07-04](../../chapters/07-04-laplace-domain-circuit-analysis.md). Handout: [README.md](README.md).

Symbolic s-domain algebra (SymPy) on one side, time-domain simulation (SciPy) on the other: every s-domain answer is checked against a simulation. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
from pathlib import Path
import numpy as np
import sympy as sp
from scipy import signal
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

s = sp.symbols("s")
HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)


def ngspice_values(*paths):
    vals = {}
    for path in paths:
        out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
        for line in out.splitlines():
            parts = line.split("=")
            if len(parts) >= 2:
                try:
                    vals[parts[0].strip()] = float(parts[1].split()[0])
                except (ValueError, IndexError):
                    pass
    return vals


NG = ngspice_values(*sorted(str(p) for p in Path(".").glob("*.cir"))) if HAVE_NGSPICE else {}

ngspice available: True


## Part 1 — The determinant is the characteristic polynomial

The third-order filter of Example 07-04.2, written as a nodal matrix in $s$.

**🔮 Predict:** how many roots does $\det\mathbf{Y}(s)$ have, and do they depend on which node is the output?

In [2]:
R = 50
wc = 2 * sp.pi * 10**6
C1 = C2 = 1 / (wc * R)
L = 2 * R / wc
Y = sp.Matrix([[1 / R + s * C1 + 1 / (s * L), -1 / (s * L)],
               [-1 / (s * L), s * C2 + 1 / R + 1 / (s * L)]])
det = sp.factor(sp.together(Y.det()))
num, den = sp.fraction(det)
roots = [complex(r) for r in sp.Poly(sp.expand(num), s).nroots()]
print("roots of det Y (Mrad/s):", [f"{r.real / 1e6:.3f}{r.imag / 1e6:+.3f}j" for r in roots])
# the same circuit as state equations (v1, iL, v2): eigenvalues
Cn, Ln = float(C1), float(L)
A = np.array([[-1 / (R * Cn), -1 / Cn, 0], [1 / Ln, 0, -1 / Ln], [0, 1 / Cn, -1 / (R * Cn)]])
print("eigenvalues of A (Mrad/s):", np.round(np.sort_complex(np.linalg.eigvals(A)) / 1e6, 3))
assert np.allclose(np.sort_complex(np.array(roots)), np.sort_complex(np.linalg.eigvals(A)), rtol=1e-6)
# transfer functions to node 1 and node 2 share the denominator: both blow up at every pole
Vs = sp.symbols("Vs")
V = Y.LUsolve(sp.Matrix([Vs / R, 0]))
Ln, Cn2 = float(L), float(C2)


def node_voltages(sv):
    Yn = np.array([[1 / R + sv * Cn + 1 / (sv * Ln), -1 / (sv * Ln)],
                   [-1 / (sv * Ln), sv * Cn2 + 1 / R + 1 / (sv * Ln)]], complex)
    return np.linalg.solve(Yn, np.array([1 / R, 0], complex))


for p in np.linalg.eigvals(A):
    v1p, v2p = node_voltages(p * (1 + 1e-7))
    print(f"near the pole {p / 1e6:.3f} Mrad/s: |V1/Vs| = {abs(v1p):.2e}, |V2/Vs| = {abs(v2p):.2e}")
    assert abs(v1p) > 1e5 and abs(v2p) > 1e5
if NG:
    print("ngspice .pz:", NG["ex2pr"], NG["ex2cr"], "+/- j", NG["ex2ci"])

roots of det Y (Mrad/s): ['-6.283+0.000j', '-3.142-5.441j', '-3.142+5.441j']
eigenvalues of A (Mrad/s): [-6.283+0.j    -3.142-5.441j -3.142+5.441j]
near the pole -6.283+0.000j Mrad/s: |V1/Vs| = 5.00e+06, |V2/Vs| = 5.00e+06
near the pole -3.142+5.441j Mrad/s: |V1/Vs| = 2.89e+06, |V2/Vs| = 2.89e+06
near the pole -3.142-5.441j Mrad/s: |V1/Vs| = 2.89e+06, |V2/Vs| = 2.89e+06
ngspice .pz: -6283190.0 -3141590.0 +/- j 5441398.0


Three roots, the same as the state matrix's eigenvalues, and both node voltages blow up at every one of them: the poles belong to the circuit. Only the numerators differ.

## Part 2 — A switched circuit, transformed

Example 07-04.1 (07-03 Problem 07-03.7).

**🔮 Predict:** the initial and final values of $v_C$, from the limit theorems alone.

In [3]:
Rr, Lr, Cr = 2, sp.Rational(1, 2000), sp.Rational(50, 10**6)
iL0, vC0 = 2, 8
I = (sp.Rational(12) / s + Lr * iL0 - sp.Rational(vC0) / s) / (Rr + s * Lr + 1 / (s * Cr))
VC = sp.factor(I / (s * Cr) + sp.Rational(vC0) / s)
print("VC(s) =", VC)
print("IVT:", sp.limit(s * VC, s, sp.oo), " FVT:", sp.limit(s * VC, s, 0))
print("partial fractions:", sp.apart(VC, s))
A = np.array([[-2 / 0.5e-3, -1 / 0.5e-3], [1 / 50e-6, 0.0]])
sol = solve_ivp(lambda t, x: A @ x + np.array([12 / 0.5e-3, 0]), (0, 3e-3), [2.0, 8.0],
                rtol=1e-10, atol=1e-12, dense_output=True, max_step=1e-6)
tt = np.linspace(0, 3e-3, 3001)
vc_s = 12 + np.exp(-2000 * tt) * (-4 * np.cos(6000 * tt) + 16 / 3 * np.sin(6000 * tt))
print(f"v_C(0.5 ms): s-domain {12 + np.exp(-1) * (-4 * np.cos(3) + 16 / 3 * np.sin(3)):.4f} V, simulation {sol.sol(0.5e-3)[1]:.4f} V")
assert np.allclose(sol.sol(tt)[1], vc_s, atol=1e-6)
if NG:
    print("ngspice:", NG["ex1v"], "V")

VC(s) = 8*(s**2 + 9000*s + 60000000)/(s*(s**2 + 4000*s + 40000000))
IVT: 8  FVT: 12
partial fractions: -4*(s - 6000)/(s**2 + 4000*s + 40000000) + 12/s
v_C(0.5 ms): s-domain 13.7337 V, simulation 13.7337 V


ngspice: 13.73367 V


## Part 3 — Moving a zero

Example 07-04.3: the poles stay at $-5500 \pm j8930$ s⁻¹ and the DC gain at 1/11; only the zero moves.

**🔮 Predict:** how does the peak change as the zero moves from −1000 s⁻¹ to −20 000 s⁻¹?

In [4]:
den = [1, 11000, 1.1e8]
t = np.linspace(0, 1.5e-3, 30001)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
peaks = {}
for z in (1000, 3000, 20000, None):
    num = [1.1e8 / 11] if z is None else np.polymul([1 / z, 1], [1.1e8 / 11])
    _, y = signal.step((num, den), T=t)
    y *= 11
    peaks[z] = y.max()
    ax.plot(t * 1e3, y, label="all-pole" if z is None else f"zero at −{z} s⁻¹")
ax.set_xlabel("time (ms)"); ax.set_ylabel("v (V), 11 V step"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.show()
for z, pk in peaks.items():
    print(f"{'all-pole' if z is None else f'zero at -{z}':>14}: peak {pk:.3f} V")
assert abs(peaks[1000] - 6.066) < 1e-3 and abs(peaks[20000] - 1.173) < 1e-3 and abs(peaks[None] - 1.144) < 1e-3

 zero at -1000: peak 6.066 V
 zero at -3000: peak 2.390 V
zero at -20000: peak 1.173 V
      all-pole: peak 1.144 V


The zero adds $1/z$ times the derivative of the all-pole response. Near the origin it dominates; far beyond the poles it barely matters, and the response approaches the all-pole one.

## Part 4 — Crossing the imaginary axis

The Wien bridge of Example 07-04.6, with the amplifier gain $K$ as the knob.

**🔮 Predict:** the gain at which the poles reach the imaginary axis, and the frequency there.

In [5]:
Rw, Cw = 10e3, 10e-9


def wien(K):
    return np.array([[-1, K - 1], [-1, K - 2]]) / (Rw * Cw)


for K in (2.0, 2.9, 3.0, 3.1):
    e = np.linalg.eigvals(wien(K))
    print(f"K = {K}: poles {e[0].real:9.1f} ± j{abs(e[0].imag):7.1f} s^-1")
K_cross = [K for K in np.linspace(2.5, 3.5, 100001) if max(np.linalg.eigvals(wien(K)).real) > 0][0]
f_cross = abs(np.linalg.eigvals(wien(3.0))[0].imag) / (2 * np.pi)
print(f"poles cross the axis at K = {K_cross:.4f}, at {f_cross:.1f} Hz")
assert abs(K_cross - 3) < 1e-4 and abs(f_cross - 1591.5) < 0.1
t = np.linspace(0, 20e-3, 20001)
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for K in (2.9, 3.0, 3.1):
    x = solve_ivp(lambda _t, x: wien(K) @ x, (0, 20e-3), [0, 1e-3], t_eval=t, rtol=1e-9, atol=1e-12, max_step=2e-5).y[1]
    ax.plot(t * 1e3, x * 1e3, label=f"K = {K}", lw=0.8)
ax.set_xlabel("time (ms)"); ax.set_ylabel("node x (mV)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.show()
if NG:
    print("ngspice .pz at K = 2.9:", NG["ex6r"], "± j", NG["ex6i"])

K = 2.0: poles   -5000.0 ± j 8660.3 s^-1
K = 2.9: poles    -500.0 ± j 9987.5 s^-1
K = 3.0: poles      -0.0 ± j10000.0 s^-1
K = 3.1: poles     500.0 ± j 9987.5 s^-1


poles cross the axis at K = 3.0000, at 1591.5 Hz


ngspice .pz at K = 2.9: -500.0 ± j 9987.492


## Your turn

1. In Part 1, take the output at node 1 instead of node 2. What changes in the transfer function, and what does not?
2. In Part 3, put the zero in the right half-plane, at +3000 s⁻¹ (keep the DC gain). What does the step response do first?
3. In Part 4, find the gain $K$ that gives the Wien circuit a damping ratio of 0.1, and its ringing frequency.

<details><summary>Answers</summary>

1. The numerator changes: node 1's transfer function gains a pair of zeros where the branch from node 1 to ground through $L$, $L$ in series with $C_2 \parallel R$, has zero impedance: $s^2LC_2 + sL/R + 1 = 0$, that is $s = \omega_c(-0.5 \pm j0.5)$ = **−3.142 ± j3.142 Mrad/s**. The denominator, and so the poles, do not change.
2. It starts the wrong way: with $H = (1 - s/3000)H_{ap}$ the response is $y_{ap} - y_{ap}'/3000$, so it first goes negative, to **−1.50 V**, before rising to 1 V.
3. ζ = $(3 - K)/2 = 0.1$ gives **K = 2.8**; the poles are $-1000 \pm j9950$ s⁻¹, ringing at about **1.584 kHz**.

</details>

In [6]:
# Check the 'Your turn' answers.
zeros_n1 = np.roots([Ln * Cn2, Ln / R, 1])          # where sL + 1/(sC2 + 1/R) = 0
print("node-1 zeros (Mrad/s):", np.round(zeros_n1 / 1e6, 3))
# check: solve the nodal equations numerically at that s; V1 vanishes while V2 does not
v1z, v2z = node_voltages(zeros_n1[0])
print(f"at the zero: |V1/Vs| = {abs(v1z):.2e}, |V2/Vs| = {abs(v2z):.3f}")
assert abs(v1z) < 1e-9 * abs(v2z)
assert np.allclose(zeros_n1.real, -np.pi * 1e6, rtol=1e-9) and np.allclose(abs(zeros_n1.imag), np.pi * 1e6, rtol=1e-9)
num_rhp = np.polymul([-1 / 3000, 1], [1.1e8 / 11])
_, y = signal.step((num_rhp, [1, 11000, 1.1e8]), T=np.linspace(0, 1.5e-3, 30001))
print(f"RHP zero: minimum {11 * y.min():.2f} V")
assert abs(11 * y.min() + 1.50) < 0.005
e = np.linalg.eigvals(wien(2.8))
print(f"K = 2.8: zeta = {-e[0].real / abs(e[0]):.3f}, ringing {abs(e[0].imag) / (2 * np.pi):.1f} Hz")
assert abs(-e[0].real / abs(e[0]) - 0.1) < 1e-9 and abs(abs(e[0].imag) / (2 * np.pi) - 1583.6) < 0.5

node-1 zeros (Mrad/s): [-3.142+3.142j -3.142-3.142j]
at the zero: |V1/Vs| = 8.85e-16, |V2/Vs| = 1.414
RHP zero: minimum -1.50 V
K = 2.8: zeta = 0.100, ringing 1583.6 Hz
